In [17]:
import torch, torchvision
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import DataLoader,Subset

In [18]:
t = T.Compose([T.Resize((32,32)), T.ToTensor()])
trd = DataLoader(Subset(torchvision.datasets.MNIST('./data',True,transform=t,download=True),range(2000)),64,True)
ted = DataLoader(Subset(torchvision.datasets.MNIST('./data',False,transform=t,download=True),range(500)),64)


In [19]:
class LeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,6,5), nn.ReLU(), nn.AvgPool2d(2),
            nn.Conv2d(6,16,5), nn.ReLU(), nn.AvgPool2d(2),
            nn.Flatten(), nn.Linear(400,120), nn.ReLU(),nn.Linear(120,84), nn.ReLU(), nn.Linear(84,10)
        )
    def forward(self,x):
        return self.net(x)
        

In [20]:
model = LeNet()
opt = optim.Adam(model.parameters(),0.01)
crit = nn.CrossEntropyLoss()
for e in range(5):
    rl=0
    model.train()
    for x,y in trd:
        opt.zero_grad()
        o = model(x)
        l = crit(o,y)
        l.backward()
        rl += l.item()
        opt.step()
    print(f'Loss {e+1}: {rl/len(trd)}')
n=c=0
with torch.no_grad():
    for x,y in ted:
        o = model(x)
        c += (o.argmax(1)==y).sum().item()
        n += len(y)
print(f'Accuracy: {c/n*100:.2f}%')


Loss 1: 1.291955666616559
Loss 2: 0.4058581399731338
Loss 3: 0.22728553297929466
Loss 4: 0.15048648859374225
Loss 5: 0.09481705346843228
Accuracy: 95.80%
